In [ ]:
# skeleton vibed using claude
import optuna
import xgboost as xgb
import lightgbm as lgb
import numpy as np
from sklearn.metrics import root_mean_squared_error


def tune_xgb(df_train, df_test, features, n_trials=100):
    X_train = df_train[features]
    y_train = df_train['target']
    X_test  = df_test[features]
    y_test  = df_test['target']

    def objective(trial):
        params = {
            "device":                   "cuda",
            "objective":                "reg:squarederror",
            "n_estimators":             5000,
            "verbosity":                0,
            "enable_categorical":       True,
            "tree_method":              'hist',
            "max_depth":                trial.suggest_int("max_depth", 6, 9),
            "min_child_weight":         trial.suggest_int("min_child_weight", 10, 50),
            
            "learning_rate":            trial.suggest_float("learning_rate", 0.008, 0.02, log=True),
            "subsample":                trial.suggest_float("subsample", 0.7, 0.95),
            "colsample_bytree":         trial.suggest_float("colsample_bytree", 0.3, 0.8),
            "colsample_bylevel":        trial.suggest_float("colsample_bylevel", 0.3, 0.8),
            
            "reg_alpha":                trial.suggest_float("reg_alpha", 0.1, 10.0, log=True), 
            "reg_lambda":               trial.suggest_float("reg_lambda", 20.0, 100.0, log=True),
            "gamma":                    trial.suggest_float("gamma", 0.5, 2.0),
            "early_stopping_rounds":    100,
            "random_state":             42,
        }

        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False
        )

        preds = model.predict(X_test)
        return root_mean_squared_error(y_test, preds)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print("\n=== XGB Best trial ===")
    print(f"  RMSE : {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")

    # Retrain with best params on full data
    best = study.best_params
    best_model = xgb.XGBRegressor(
        device="cuda",
        objective="reg:squarederror",
        enable_categorical=True,
        tree_method='hist',
        n_estimators=5000,
        early_stopping_rounds=100,
        random_state=42,
        **best
    )
    best_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )

    return best_model, study


def tune_lgbm(df_train, df_test, features, n_trials=100):
    X_train = df_train[features]
    y_train = df_train['target']
    X_test  = df_test[features]
    y_test  = df_test['target']

    def objective(trial):
        params = {
            "objective":            "regression",
            "metric":               "rmse",
            "n_estimators":         5000,
            "verbosity":            -1,
            "random_state":         42,

            "num_leaves":           trial.suggest_int("num_leaves", 32, 128),
            "max_depth":            trial.suggest_int("max_depth", 6, 12),
            
            "learning_rate":        trial.suggest_float("learning_rate", 0.008, 0.03, log=True),
            
            "bagging_fraction":     trial.suggest_float("bagging_fraction", 0.7, 0.95), # Same as subsample
            "bagging_freq":         trial.suggest_int("bagging_freq", 1, 5),
            "feature_fraction":     trial.suggest_float("feature_fraction", 0.1, 0.5), # Same as colsample_bytree

            "min_split_gain":       trial.suggest_float("min_split_gain", 0.5, 2.0),
            
            "lambda_l1":            trial.suggest_float("lambda_l1", 0.1, 10.0, log=True),
            
            "lambda_l2":            trial.suggest_float("lambda_l2", 10.0, 50.0, log=True),
            
            "min_child_samples":    trial.suggest_int("min_child_samples", 20, 100),
        }

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="rmse",
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)   # silences per-iter output
            ]
        )

        preds = model.predict(X_test)
        return root_mean_squared_error(y_test, preds)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print("\n=== LGBM Best trial ===")
    print(f"  RMSE : {study.best_value:.4f}")
    print(f"  Params: {study.best_params}")

    # Retrain with best params
    best = study.best_params
    best_model = lgb.LGBMRegressor(
        n_estimators=3000,
        random_state=42,
        **best
    )
    best_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(period=100)
        ]
    )

    return best_model, study



In [6]:
from dataengineers import Dataset

ds = Dataset('train')
train, test = ds.build_train_test()
features = [c for c in train.columns if c not in ['target', 'id', 'delivery_start']]

In [7]:
xgb_model, xgb_study = tune_xgb(train, test, features, n_trials=100)

[I 2026-02-22 14:43:40,114] A new study created in memory with name: no-name-4c507ddb-10c4-4671-941b-54aa215d8dd3
Best trial: 0. Best value: 44.4838:   1%|          | 1/100 [00:04<07:27,  4.52s/it]

[I 2026-02-22 14:43:44,630] Trial 0 finished with value: 44.48381455757348 and parameters: {'max_depth': 9, 'min_child_weight': 19, 'learning_rate': 0.013110883870132347, 'subsample': 0.9412876278075754, 'colsample_bytree': 0.6548908061550665, 'colsample_bylevel': 0.45844498492046837, 'reg_alpha': 0.16589510046347383, 'reg_lambda': 87.29498165866009, 'gamma': 1.129682193322878}. Best is trial 0 with value: 44.48381455757348.


Best trial: 0. Best value: 44.4838:   2%|▏         | 2/100 [00:08<06:51,  4.19s/it]

[I 2026-02-22 14:43:48,599] Trial 1 finished with value: 46.02124182949175 and parameters: {'max_depth': 9, 'min_child_weight': 29, 'learning_rate': 0.012278783154806506, 'subsample': 0.9186622150769392, 'colsample_bytree': 0.50217055289635, 'colsample_bylevel': 0.6575562064171205, 'reg_alpha': 2.2088874140413126, 'reg_lambda': 66.24728240406085, 'gamma': 1.763156252209679}. Best is trial 0 with value: 44.48381455757348.


Best trial: 0. Best value: 44.4838:   3%|▎         | 3/100 [00:11<05:51,  3.62s/it]

...

[I 2026-02-22 14:52:05,472] Trial 97 finished with value: 43.02881600060922 and parameters: {'max_depth': 9, 'min_child_weight': 10, 'learning_rate': 0.009685288825301714, 'subsample': 0.7805382187905897, 'colsample_bytree': 0.31288557430454733, 'colsample_bylevel': 0.6930663807077051, 'reg_alpha': 0.1780114684050397, 'reg_lambda': 66.48851696398607, 'gamma': 0.9857037728927389}. Best is trial 95 with value: 42.701795618094366.


Best trial: 95. Best value: 42.7018:  99%|█████████▉| 99/100 [08:32<00:06,  6.79s/it]

[I 2026-02-22 14:52:12,225] Trial 98 finished with value: 42.9393742136821 and parameters: {'max_depth': 9, 'min_child_weight': 10, 'learning_rate': 0.010596851105086649, 'subsample': 0.7701154149695294, 'colsample_bytree': 0.3304984281378433, 'colsample_bylevel': 0.6774889003141872, 'reg_alpha': 0.1699632885115349, 'reg_lambda': 73.75120090792063, 'gamma': 1.2051092364425913}. Best is trial 95 with value: 42.701795618094366.


Best trial: 95. Best value: 42.7018: 100%|██████████| 100/100 [08:40<00:00,  5.20s/it]


[I 2026-02-22 14:52:20,318] Trial 99 finished with value: 42.86093021723486 and parameters: {'max_depth': 9, 'min_child_weight': 10, 'learning_rate': 0.010588745669675086, 'subsample': 0.7701664581443386, 'colsample_bytree': 0.33511777143507376, 'colsample_bylevel': 0.6828673628700643, 'reg_alpha': 0.16356330080914205, 'reg_lambda': 77.8662032173368, 'gamma': 1.0225934933626417}. Best is trial 95 with value: 42.701795618094366.

=== XGB Best trial ===
  RMSE : 42.7018
  Params: {'max_depth': 9, 'min_child_weight': 10, 'learning_rate': 0.011041791784948136, 'subsample': 0.769216997688628, 'colsample_bytree': 0.3002874876389264, 'colsample_bylevel': 0.68925894223455, 'reg_alpha': 0.17428426292668459, 'reg_lambda': 77.5689626698248, 'gamma': 0.9947338308302472}
[0]	validation_0-rmse:62.87876
[100]	validation_0-rmse:53.46553
[200]	validation_0-rmse:49.32961
[300]	validation_0-rmse:47.05757
[400]	validation_0-rmse:45.53266
[500]	validation_0-rmse:44.48206
[600]	validation_0-rmse:43.80065
[7

In [4]:
lgb_model, lgb_study  = tune_lgbm(train, test, features, n_trials=100)

[I 2026-02-22 14:32:20,105] A new study created in memory with name: no-name-e541eeb5-457e-43e7-839e-ed33d2d41f96
Best trial: 0. Best value: 50.6057:   1%|          | 1/100 [00:04<06:41,  4.06s/it]

[I 2026-02-22 14:32:24,161] Trial 0 finished with value: 50.605688091583204 and parameters: {'num_leaves': 110, 'max_depth': 6, 'learning_rate': 0.012566502713598639, 'bagging_fraction': 0.9240551710272266, 'bagging_freq': 4, 'feature_fraction': 0.1960202316379406, 'min_split_gain': 1.504919701666347, 'lambda_l1': 6.199809225686039, 'lambda_l2': 41.87268842227375, 'min_child_samples': 64}. Best is trial 0 with value: 50.605688091583204.


Best trial: 1. Best value: 47.6806:   2%|▏         | 2/100 [00:07<05:45,  3.53s/it]

[I 2026-02-22 14:32:27,316] Trial 1 finished with value: 47.68060259562326 and parameters: {'num_leaves': 45, 'max_depth': 6, 'learning_rate': 0.025876646416252508, 'bagging_fraction': 0.8272648395385495, 'bagging_freq': 1, 'feature_fraction': 0.10579075471321508, 'min_split_gain': 0.5294235033354453, 'lambda_l1': 0.9533301682638248, 'lambda_l2': 43.679176656886725, 'min_child_samples': 87}. Best is trial 1 with value: 47.68060259562326.


Best trial: 1. Best value: 47.6806:   3%|▎         | 3/100 [00:11<06:01,  3.72s/it]

...

[I 2026-02-22 14:39:20,305] Trial 97 finished with value: 41.912528178813844 and parameters: {'num_leaves': 122, 'max_depth': 12, 'learning_rate': 0.024877649144486133, 'bagging_fraction': 0.7917338215573899, 'bagging_freq': 1, 'feature_fraction': 0.13330027457816165, 'min_split_gain': 1.125119076921349, 'lambda_l1': 2.8794952865352235, 'lambda_l2': 16.988514191627697, 'min_child_samples': 22}. Best is trial 74 with value: 41.26499340338209.


Best trial: 74. Best value: 41.265:  99%|█████████▉| 99/100 [07:02<00:03,  3.01s/it]

[I 2026-02-22 14:39:22,787] Trial 98 finished with value: 47.505510358370316 and parameters: {'num_leaves': 117, 'max_depth': 12, 'learning_rate': 0.025960324675865388, 'bagging_fraction': 0.7769580309559007, 'bagging_freq': 1, 'feature_fraction': 0.38755199107888194, 'min_split_gain': 0.9258702002636515, 'lambda_l1': 1.0079334307619177, 'lambda_l2': 16.198384460385267, 'min_child_samples': 30}. Best is trial 74 with value: 41.26499340338209.


Best trial: 74. Best value: 41.265: 100%|██████████| 100/100 [07:05<00:00,  4.25s/it]


[I 2026-02-22 14:39:25,410] Trial 99 finished with value: 43.375796181768145 and parameters: {'num_leaves': 114, 'max_depth': 12, 'learning_rate': 0.023786092698004736, 'bagging_fraction': 0.7661120106339524, 'bagging_freq': 1, 'feature_fraction': 0.15221878629637553, 'min_split_gain': 1.0153832197260428, 'lambda_l1': 3.2600763759963836, 'lambda_l2': 20.12221508448583, 'min_child_samples': 33}. Best is trial 74 with value: 41.26499340338209.

=== LGBM Best trial ===
  RMSE : 41.2650
  Params: {'num_leaves': 111, 'max_depth': 11, 'learning_rate': 0.017450649341248122, 'bagging_fraction': 0.7317287482536371, 'bagging_freq': 1, 'feature_fraction': 0.11288101765162045, 'min_split_gain': 1.0482834566823582, 'lambda_l1': 2.075968543559966, 'lambda_l2': 20.957202176792954, 'min_child_samples': 22}
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 50.5096	valid_0's l2: 2551.22
[200]	valid_0's rmse: 46.0975	valid_0's l2: 2124.98
[300]	valid_0's rmse: 44.0214	val